In [2]:
import kagglehub
import numpy as np
from torch import nn
import matplotlib.pyplot as plt 
import pandas as pd
import os
from pathlib import Path
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.io as io
from torchvision import transforms, utils

/home/kels/adv_ml/ml_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

dir = "./data"

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset", output_dir=dir + "/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database", output_dir=dir+"/covid19-radiography-database")
print("Path to dataset files:", path)

Path to dataset files: ./data/brain-tumor-mri-dataset
Path to dataset files: ./data/covid19-radiography-database


In [3]:
class scans_dataset(Dataset):
    def __init__(self, *, dataset):
        self.df = self.build_dataframe(dataset)
        self.length = self.df.shape[0]
    
    def __getitem__(self, index):
        item = self.df.iloc[index]
        scan = io.read_image(item['scan'])
        mask = io.read_image(item['mask'])
        if item['mask']:
            return scan, item['label'], mask
        else:
            return mask, item['label']
        
    
        
    def build_dataframe(self, dataset="covid"):
        df = pd.DataFrame()
        if dataset == "tumors":
            path = "data/brain-tumor-mri-dataset"
            for t in ["Training", "Testing"]:
                for f in [d for d in os.listdir(path + "/" + t)]:
                    files = [d for d in os.listdir(path + f'/{t}/{f}')]
                    df = pd.concat([df, pd.DataFrame({'testing': t=="Testing", 'label': Path(f).name, 'scan': files})], ignore_index=True)
        elif dataset == "covid":
            path = "data/covid19-radiography-database/COVID-19_Radiography_Dataset"
            for f in [d for d in os.listdir(path)]:
                images = [f'{path}/{f}/images/{d}' for d in os.listdir(path + f'/{f}/images')]
                masks = [f'{path}/{f}/masks/{d}' for d in os.listdir(path+f'/{f}/masks')]
                df = pd.concat([df, pd.DataFrame({'label': Path(f).name, 'scan': images, 'mask': masks})], ignore_index=True)
        else:
            raise ValueError("No such dataset")
        
        return df
            

In [91]:
data = scans_dataset(dataset="covid")
scan, label, mask = data.__getitem__(11)
transform = transforms.CenterCrop(size=(256,256))
scan = transform(scan)

In [5]:
class ResidualLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, n_feats, dropout, stride):
        super(ResidualLayer, self).__init__()

        self.down_scale_flag = in_channels != out_channels
        self.mish = nn.Mish()

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels = out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=1
        )

        self.down_scale = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=1,
                stride=stride
            ),
            nn.BatchNorm2d(out_channels)
        )

        self.conv_block = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            self.mish(),
            nn.Dropout(dropout),
            self.conv(),
            nn.BatchNorm2d(out_channels),
            self.mish(),
            nn.Dropout(dropout),
            self.conv()
        )
        
    def forward(self, x):
        identity = self.down_scale(x) if self.down_scale_flag else x
        x = self.conv_block(x)
        x = self.mish(x+ identity)
        return x
        

In [7]:
class scan_cnn(nn.Module):
    def __init__(self, batch_size, n_feats, n_classes, input_channels, kernel_size, dropout, stride, image_size: tuple):
        super(scan_cnn, self).__init__()
        self.conv_block = nn.Conv2d(in_channels=input_channels,
                                    out_channels=8,
                                    padding=(1,1),
                                    stride=stride)
        self.mish = nn.Mish()
        self.norm = nn.BatchNorm2d(16)
        self.residual_block_1 = nn.Sequential(
            ResidualLayer(8,16,kernel_size, n_feats, dropout, stride),
            ResidualLayer(16,16,kernel_size, n_feats, dropout, stride),
            ResidualLayer(16,16,kernel_size, n_feats, dropout, stride),
            nn.AvgPool2d(kernel_size, stride)
        )
        self.residual_block_2 = nn.Sequential(
            ResidualLayer(16, 32, kernel_size, n_feats, dropout, stride),
            ResidualLayer(32, 32, kernel_size, n_feats, dropout, stride),
            ResidualLayer(32, 32, kernel_size, n_feats, dropout, stride),
            nn.AvgPool2d(kernel_size, stride)
        )
        self.residual_block_3 = nn.Sequential(
            ResidualLayer(32, 64, kernel_size, n_feats, dropout, stride),
            ResidualLayer(64, 64, kernel_size, n_feats, dropout, stride),
            ResidualLayer(64, 64, kernel_size, n_feats, dropout, stride),
            nn.AvgPool2d(kernel_size, stride)
        )
        self.residual_block_4 = nn.Sequential(
            ResidualLayer(64, 128, kernel_size, n_feats, dropout, stride),
            ResidualLayer(128, 128, kernel_size, n_feats, dropout, stride),
            ResidualLayer(128, 128, kernel_size, n_feats, dropout, stride),
            nn.AvgPool2d(kernel_size, stride)
        )

        self.flatten = nn.Flatten(1)
        self.linear = nn.Linear(
            image_size[0] / 2**4 * image_size[1] / 2**4 * 128,
            n_classes
        )
    
    def forward(self, x):
        x = self.conv_block(x)
        x = self.norm(x)
        x = self.mish(x)

        x = self.residual_block_1(x)
        x = self.residual_block_2(x)
        x = self.residual_block_3(x)
        x = self.residual_block_4(x)

        x = self.flatten(x)
        x = self.linear(x)

        return x